# Car Price Advisor — Exploratory Notebook

Walks through EDA and modeling for a single make/model. The production training script (`car_predictor.py`) trains a global model across all makes/models and powers the web app.

**To switch cars:** change `TARGET_MAKE` and `TARGET_MODEL` below and re-run all cells.

**To run the web app:** `python car_predictor.py` then `python app.py`.

## 1 · Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings, json, joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATA_PATH = Path("vehicles.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# ---- change these two lines to analyze any car ----
TARGET_MAKE  = "toyota"
TARGET_MODEL = "camry"
# ---------------------------------------------------

CURRENT_YEAR = 2024
PRICE_MIN, PRICE_MAX = 1_500, 60_000
MILEAGE_MAX = 300_000
YEAR_MIN = 2000

print(f"Target: {TARGET_MAKE.title()} {TARGET_MODEL.title()}")
print(f"Data path exists: {DATA_PATH.exists()}")

## 2 · Load & Filter

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Full dataset: {len(df_raw):,} rows x {df_raw.shape[1]} cols")

for col in ["manufacturer","model","condition","drive","transmission","fuel","title_status"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(str).str.lower().str.strip()

df = df_raw[df_raw["manufacturer"] == TARGET_MAKE].copy()
df = df[df["model"].str.contains(TARGET_MODEL, na=False)].copy()

df["price"]    = pd.to_numeric(df["price"],    errors="coerce")
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["odometer"] = pd.to_numeric(df["odometer"], errors="coerce")

df = df[
    df["price"].between(PRICE_MIN, PRICE_MAX)
    & df["year"].between(YEAR_MIN, CURRENT_YEAR)
    & df["odometer"].between(0, MILEAGE_MAX)
].copy()

df["age"] = CURRENT_YEAR - df["year"]
print(f"Clean rows: {len(df):,}")
df[["age","price","odometer"]].describe().round(0)

## 3 · EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(f"{TARGET_MAKE.title()} {TARGET_MODEL.title()} — Listing Overview", fontweight="bold")

axes[0,0].hist(df["price"], bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
axes[0,0].set_title("Price distribution")
axes[0,0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

axes[0,1].hist(df["odometer"], bins=60, color="#55A868", edgecolor="white", linewidth=0.4)
axes[0,1].set_title("Odometer distribution")
axes[0,1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"{x/1000:.0f}k mi"))

axes[0,2].scatter(df["age"].clip(0,20), df["price"], alpha=0.15, s=6, color="#C44E52")
axes[0,2].set_title("Price vs Age")
axes[0,2].set_xlabel("Age (years)")
axes[0,2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

age_price = df.groupby("age")["price"].median().reset_index()
axes[1,0].plot(age_price["age"], age_price["price"], marker="o", markersize=4, color="#4C72B0")
axes[1,0].set_title("Median price by age")
axes[1,0].set_xlabel("Age (years)")
axes[1,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

if "condition" in df.columns:
    cc = df["condition"].value_counts()
    axes[1,1].barh(cc.index, cc.values, color="#8172B2")
    axes[1,1].set_title("Listings by condition")

top_states = df.groupby("state")["price"].median().nlargest(10)
axes[1,2].barh(top_states.index[::-1], top_states.values[::-1], color="#CCB974")
axes[1,2].set_title("Median price — top 10 states")
axes[1,2].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

plt.tight_layout()
plt.show()

## 4 · Feature Engineering

In [ ]:
CONDITION_MAP = {"salvage":1,"fair":2,"good":3,"excellent":4,"like new":5,"new":6}
DRIVE_MAP     = {"fwd":0,"rwd":1,"4wd":2}
TRANS_MAP     = {"automatic":1,"manual":0,"other":0}
FUEL_MAP      = {"gas":0,"hybrid":1,"electric":2,"diesel":3,"other":0}
TITLE_MAP     = {"clean":0,"rebuilt":2,"salvage":4,"lien":1,"missing":3,"parts only":5}

df["condition_score"] = df["condition"].map(CONDITION_MAP).fillna(3)
df["drive_enc"]       = df["drive"].map(DRIVE_MAP).fillna(0)
df["trans_enc"]       = df["transmission"].map(TRANS_MAP).fillna(1)
df["fuel_enc"]        = df["fuel"].map(FUEL_MAP).fillna(0)
df["title_risk"]      = df["title_status"].map(TITLE_MAP).fillna(0)

if "cylinders" in df.columns:
    df["cylinders_n"] = df["cylinders"].astype(str).str.extract(r"(\d+)")[0].astype(float)
df["cylinders_n"] = pd.to_numeric(df.get("cylinders_n", 4.0), errors="coerce").fillna(4.0)

global_median = df["price"].median()
state_medians = df.groupby("state")["price"].median()
df["state_ratio"] = df["state"].map(state_medians) / global_median

df["age_x_miles"] = df["age"] * df["odometer"]

FEATURES = [
    "age","odometer","condition_score","drive_enc","trans_enc",
    "fuel_enc","title_risk","cylinders_n","state_ratio","age_x_miles",
]

df[FEATURES].describe().round(2)

## 5 · Targets & Split

In [ ]:
df_model = df.dropna(subset=FEATURES + ["price"]).copy()
X          = df_model[FEATURES].values
y_purchase = df_model["price"].values

df_model["age_bucket"] = pd.cut(df_model["age"], bins=[0,3,6,9,15,25], labels=False)
y_tradein = (
    df_model.groupby(["age_bucket","condition_score"])["price"]
    .transform(lambda g: g.quantile(0.25))
    .values
)

X_tr, X_te, yp_tr, yp_te, yt_tr, yt_te = train_test_split(
    X, y_purchase, y_tradein, test_size=0.2, random_state=42
)
print(f"Train: {len(X_tr):,}  Test: {len(X_te):,}")
print(f"Purchase median train: ${np.median(yp_tr):,.0f}")
print(f"Trade-in median train: ${np.median(yt_tr):,.0f}")

## 6 · Train Models

In [ ]:
purchase_model = Pipeline([
    ("scaler", StandardScaler()),
    ("gbr", GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        min_samples_leaf=10, subsample=0.8,
        loss="absolute_error", random_state=42,
    )),
])
print("Training purchase model...")
purchase_model.fit(X_tr, yp_tr)
print("Done.")

In [ ]:
tradein_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(
        n_estimators=300, max_depth=10,
        min_samples_leaf=10, random_state=42, n_jobs=-1,
    )),
])
print("Training trade-in model...")
tradein_model.fit(X_tr, yt_tr)
print("Done.")

## 7 · Evaluate

In [ ]:
def eval_model(model, X_test, y_test, label):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / (y_test + 1e-6))) * 100
    print(f"{label}")
    print(f"  MAE ${mae:,.0f}   R2 {r2:.3f}   MAPE {mape:.1f}%")
    return preds

p_preds = eval_model(purchase_model, X_te, yp_te, "Purchase model")
t_preds = eval_model(tradein_model,  X_te, yt_te, "Trade-in model")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Predicted vs Actual — test set", fontweight="bold")

for ax, preds, y_test, label, color in [
    (axes[0], p_preds, yp_te, "Purchase Model", "#4C72B0"),
    (axes[1], t_preds, yt_te, "Trade-In Model", "#55A868"),
]:
    lim = (0, 50_000)
    ax.scatter(y_test, preds, alpha=0.15, s=5, color=color)
    ax.plot(lim, lim, "k--", linewidth=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
    ax.set_title(label)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

plt.tight_layout()
plt.show()

## 8 · Feature Importance

In [ ]:
def plot_importance(model, feature_names, title, color):
    step_name = list(model.named_steps)[-1]
    imp    = model.named_steps[step_name].feature_importances_
    ranked = sorted(zip(feature_names, imp), key=lambda x: x[1])
    feats, vals = zip(*ranked)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(feats, vals, color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()

plot_importance(purchase_model, FEATURES, "Purchase Model — Feature Importance", "#4C72B0")
plot_importance(tradein_model,  FEATURES, "Trade-In Model — Feature Importance",  "#55A868")

## 9 · Residuals

In [ ]:
residuals = yp_te - p_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Purchase Model Residuals", fontweight="bold")

axes[0].hist(residuals, bins=80, color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[0].set_xlabel("Actual - Predicted ($)")
axes[0].set_title("Residual distribution")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

axes[1].scatter(p_preds, residuals, alpha=0.15, s=5, color="#C44E52")
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicted price")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs fitted")
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))

plt.tight_layout()
plt.show()
print(f"Residual std: ${np.std(residuals):,.0f}")
print(f"Mean residual: ${np.mean(residuals):,.0f}")

## 10 · Negotiation Advisor

Fill in your car details below and run the cell.

In [ ]:
# ---- car you are buying ----
BUY_MAKE         = TARGET_MAKE
BUY_MODEL        = TARGET_MODEL
BUY_YEAR         = 2019
BUY_ODOMETER     = 55_000
BUY_CONDITION    = "good"       # salvage / fair / good / excellent / like new / new
BUY_DRIVE        = "fwd"        # fwd / rwd / 4wd
BUY_TRANSMISSION = "automatic"
BUY_FUEL         = "gas"
BUY_TITLE        = "clean"
BUY_CYLINDERS    = 4
BUY_STATE        = "ca"
DEALER_ASKING    = None         # set to sticker price if known

# ---- car you are trading in (set TRADE_MAKE = "" to skip) ----
TRADE_MAKE         = TARGET_MAKE
TRADE_MODEL        = TARGET_MODEL
TRADE_YEAR         = 2016
TRADE_ODOMETER     = 88_000
TRADE_CONDITION    = "good"
TRADE_DRIVE        = "fwd"
TRADE_TRANSMISSION = "automatic"
TRADE_FUEL         = "gas"
TRADE_TITLE        = "clean"
TRADE_CYLINDERS    = 4
DEALER_TRADE_OFFER = None

In [ ]:
COND_M  = {"salvage":1,"fair":2,"good":3,"excellent":4,"like new":5,"new":6}
DRIVE_M = {"fwd":0,"rwd":1,"4wd":2}
TRANS_M = {"automatic":1,"manual":0}
FUEL_M  = {"gas":0,"hybrid":1,"electric":2,"diesel":3}
TITLE_M = {"clean":0,"rebuilt":2,"salvage":4,"lien":1}

def make_row(year, odo, cond, drive, trans, fuel, title, cyls, state):
    age   = CURRENT_YEAR - year
    s_rat = state_medians.get(state.lower(), global_median) / global_median
    return np.array([[
        age, odo,
        COND_M.get(cond, 3), DRIVE_M.get(drive, 0),
        TRANS_M.get(trans, 1), FUEL_M.get(fuel, 0),
        TITLE_M.get(title, 0), float(cyls), s_rat, age * odo,
    ]])

buy_X    = make_row(BUY_YEAR, BUY_ODOMETER, BUY_CONDITION, BUY_DRIVE,
                    BUY_TRANSMISSION, BUY_FUEL, BUY_TITLE, BUY_CYLINDERS, BUY_STATE)
buy_pred = float(purchase_model.predict(buy_X)[0])
p_std    = float(np.std(yp_te - p_preds))
buy_lo   = max(1_500, buy_pred - p_std * 0.8)
buy_hi   = buy_pred + p_std * 0.6
target   = (buy_lo + buy_hi) / 2

SEP = "=" * 52
print(SEP)
print(f" {BUY_MAKE.upper()} {BUY_MODEL.upper()} DEAL ADVISOR")
print(SEP)
print(f"\n BUYING")
print(f"  Fair range  ${buy_lo:>9,.0f}  -  ${buy_hi:,.0f}")
if DEALER_ASKING:
    pct  = (DEALER_ASKING - target) / target * 100
    flag = "HIGH" if pct > 10 else ("slightly high" if pct > 3 else "fair")
    print(f"  Dealer asks ${DEALER_ASKING:>9,.0f}  ({pct:+.0f}% vs fair  {flag})")
print(f"  Open offer  ${buy_lo*0.93:>9,.0f}")
print(f"  Target      ${target:>9,.0f}")
print(f"  Walk away   ${buy_hi*1.02:>9,.0f}")

if TRADE_MAKE:
    trade_X  = make_row(TRADE_YEAR, TRADE_ODOMETER, TRADE_CONDITION, TRADE_DRIVE,
                        TRADE_TRANSMISSION, TRADE_FUEL, TRADE_TITLE, TRADE_CYLINDERS, BUY_STATE)
    raw      = float(tradein_model.predict(trade_X)[0])
    t_pred   = raw * 0.85
    t_std    = float(np.std(yt_te - t_preds))
    trade_lo = max(500, t_pred - t_std * 0.5)
    trade_hi = t_pred + t_std * 0.4
    trade_mid = (trade_lo + trade_hi) / 2
    print(f"\n TRADE-IN  ({TRADE_MAKE.upper()} {TRADE_MODEL.upper()})")
    print(f"  Expected  ${trade_lo:>9,.0f}  -  ${trade_hi:,.0f}")
    if DEALER_TRADE_OFFER:
        gap = trade_mid - DEALER_TRADE_OFFER
        tag = f"push back, worth ${gap:,.0f} more" if gap > 0 else "within range"
        print(f"  Offered   ${DEALER_TRADE_OFFER:>9,.0f}  ({tag})")
    print(f"  Ask for   ${trade_hi*1.05:>9,.0f}")
    print(f"  Floor     ${trade_lo*0.90:>9,.0f}")
    print(f"\n COMBINED net  approx ${target - trade_mid:,.0f}")

print(f"\n  Tip: negotiate purchase price and trade-in separately.")
print(SEP)